# Upstream Test4Py × Qwen Coder 14B NF4 — gated paired pilot
Enable **Internet** and **T4 x2**. Upload this notebook and Run All.

Both baseline and improved run pinned upstream call graph, summaries, type inference,
RAG, test generation and adaptive repair using the SAME 14B NF4 model. No 4B Critic.
Greedy initialization has an exact-prompt shared cache; sampled seeds vary generation/repair.
Compare generation costs separately from cache-dependent initialization costs.
Shared runtime adapters: CPU embedding, deterministic complete-block extraction and
empty-candidate retrieval handling. Improved adds verified type context/imports and
assertion-preserving acceptance. This is a small adapted reproduction, not the paper's
GPT-4o result or a held-out generalization study.

Order: source tests → prepare → CPU fixture through real upstream → forced-decode
VRAM stress on each GPU → small A/B run. It stops after small for artifact review.
Larger run is an explicit optional cell and refuses incomplete small/stress evidence.
Runtime is measured, not promised. All failed outcomes are retained.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, io, json, os, shutil, subprocess, sys, zipfile
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORKDIR=Path('/kaggle/working')/('Test4Py_upstream_'+STAMP)
ROOT=Path('/kaggle/working')/('UPSTREAM_NF4_'+STAMP)
subprocess.run(['git','clone','--depth','1','https://github.com/nguyendevwk/Test4Py-Setup-pilot-temp.git',str(WORKDIR)],check=True)
BASE_COMMIT='4562fa57cd5e59c0a066c1694b0b78c70437cd31'
if subprocess.check_output(['git','-C',str(WORKDIR),'rev-parse','HEAD'],text=True).strip()!=BASE_COMMIT:
    subprocess.run(['git','-C',str(WORKDIR),'fetch','--depth','1','origin',BASE_COMMIT],check=True)
    subprocess.run(['git','-C',str(WORKDIR),'checkout','--detach','FETCH_HEAD'],check=True)
assert subprocess.check_output(['git','-C',str(WORKDIR),'rev-parse','HEAD'],text=True).strip()==BASE_COMMIT
HASHES={'ablation_runner.py': '2e139f4b8f083cb7d83157326ee39ed451d93fb5765a1facca0c55c95864154a', 'analyze.py': '50f8d6c65b26922b2df1a453317bc1d6bc1e279614c810404d7a389ffd3946dd', 'benchmark19.py': 'ad74c1b19dde7d6a09f3417040a79032eda8af18d8288f56feccc69f5fa840d1', 'benchmark_worker.py': '0025c4a74c11c8cc0c3ddef6d9f1877cbc3ea9f443be258a134fb31ef3ebeb55', 'bootstrap_kaggle.py': 'd432da528c8bbcf92647bfe09c734ac8c638e051a43a7eac0307cce42e44735c', 'bootstrap_modern.py': '3c591e4399847fa5d556e507ef8177fd5dbb9e2e8fd4049b5b3afdac9628fed5', 'check_environment.py': '7feb33cb0ebe2fa58ad87f6b2131ccbe766c8bf477c3c958d10b42af92668ef2', 'council/__init__.py': '329298564fa761413630b872e45b49d744051c50f47cc06d31edf3fcb3e8c773', 'council/critic.py': 'bcc350429209f03ef199d1f5d694d16c36b9165504dbc317bbbc1cadfc5468f9', 'council/engine.py': '655b0c407c72dc7939176b85c445f5ca9034eb4dbdbfb1c80077827ce3d2359b', 'council/generator.py': '15cceff340e99561a1b639b65ef26229cabbe4418abfcbb57ee1cb3c5aa982c3', 'council/runner.py': '2cbbdfadaa55740bc1d32835d21f4fd097433127ac28033c93fc69c573a2d9d4', 'council_multitarget.py': '6e339ead3d708a477629940412a7fd1aff2425ab3a9d4e18f8d0d28030b559a0', 'council_runner.py': 'ae2565edc07abc511f2c606f54ecdc3dc52b98c5254f4ccb0c43d150a780ecab', 'gpu_context_probe.py': 'e14bed8d7c8d6e68771e7d33ebe4f249d805ca8f5c883cee86c322db26268c88', 'kaggle_run.py': 'db922dd85c6358f72a26e027831809f87d544cd0af88e073bee0fec3531e4de1', 'notebooks/kaggle_multitarget_smoke_v4.ipynb': '788e0f93051164c82c339218b064550ae3cc845a751fb3a6a4e146601157dafe', 'paired_study.py': '823294e2c8018716c388eb4b4ce1bcb37809d5acfa08bea8e0f789a94eb2dd94', 'pilot/__init__.py': '09b67cf5fee1772b0d9f5dacbbe62d7ab6b11d6e4cc28f8f4266f6b2e0e52464', 'pilot/backend.py': 'd8ebf2f7b4cf31b837713aaf9a7b67147d888099b462906a46a0f02dc4302ec9', 'pilot/benchmark.py': 'dd0a38d738bd15793fcbc3d446f21ed2be27a0bfa163575f499cc7ed6700f374', 'pilot/common.py': '8a8e63399faed17a274fdc9328471292acca7de61de8754a022c86af3bc8d7d5', 'pilot/console.py': '92cf916183f31aa034e63ef5ed71bd617edd3a7d8bc0b048988c36bb51c4e81e', 'pilot/grounding.py': 'fe9be1949741591bcaffaacd337d75a4254ed1046f064b17f408c6463b624fa8', 'pilot/initialization_cache.py': '1e5caf4d3e4237308073a62b2ef8cfb3276798e7e20d0d48db96ed47fd3d64ec', 'pilot/instrumentation.py': '495d5607773baf30cc5d87e211a996bb6df0f5651a2a956e7157e44d0709b930', 'pilot/model_profiles.py': '05a81e1e91f99f8fafa92d4c2e2f98f25a3e0e251c302ab46bf091d697d11083', 'pilot/module_mutation.py': 'f0d8ee86287f150f370ca25e4f92253ed663e01969ce7ef2f5966bea10d7e017', 'pilot/multitarget.py': '9f061ddaf7397a647a29a47cef30331e5aa1e8f23f6d307856619bf9085500bc', 'pilot/notebook.py': '5d94be4daa0eb2056ec9aa94d272c72915529795b215078c32d70cd3d51c42ff', 'pilot/prescreen.py': '716a2e8f9b3d4ef913b5dbe5c48670673ffb184064ff094eb907cbb0a2ae19c2', 'pilot/probe.py': '367015a088a092aaec2021cc5f49796eae9cc6d15e7de16ed49ce4732d811c00', 'pilot/progress.py': 'bc0b4b0ec1997f7cdba25cdc401a3eced75e87c0d53ac801079f0ca5e186457c', 'pilot/quality.py': '92d31fb233a8102313368f251e83c120bd887ae81e423347bd9d39b303f82df7', 'pilot/runtime_limits.py': 'e12569b20cf32a936ce2021a91306c5a1738c140eee8aca59d5f5f9513afc0bf', 'pilot/scheduling.py': '3b781e64f28872a5ea792d9c12eda47caa0b271a331c75191d58fc1a9a618861', 'pilot/scoring.py': '66b847795ce11523fbda9cd49c4347ac5d29be37fcddc3e18e07ed7c56a1dd5d', 'pilot/screen_trace.py': '77feb353b6f2952b91336c978481014d7e55bd1bccc633df57ce25cac6e6bcd2', 'pilot/setup.py': '9b1f1e579a6a0526f3e92813faa0e3b2a8d6c925a5f59aca22c56d69b52fb5a9', 'pilot/telemetry.py': 'de151bd6880595c98a3bfd2cbbc1e0b26d7a13079589df70340fe1857886150f', 'pilot/upstream_adapter.py': 'dfef842a9fe5e0b8db5054da2ee77b2e45273470ae2462379fb8f7c8367dcb17', 'pilot/validate.py': 'd1e57a63496b5985c182a56025c083d3778d299f3d3cf5ba136a8710b9b890f3', 'pilot.json': 'af87638ffa25424041c9b172e11aaa160e6efce34ffcb50dad7953d86187a210', 'pytest.ini': '0d6e34350f9e9a731b52183ee69d00334a8c560d849ccc9278b1ebfe48c73a9d', 'requirements-analysis.txt': '6fe163ca5631bab1ca772c8ea624297c2b3b5b86b40d9059e7ceebf6eb717510', 'requirements-modern.txt': '97c174f60ea4a808b2ed8cfb477bcc9188df1aedc65802012cbdbfce82657fa4', 'requirements-notebook.txt': '9847fe4dc03a9a4f1ca1752873d1ed97035c75ad21f699b68fcca811d366f879', 'requirements-paired.txt': 'e603a5074d7ccd0531e5c37ebb731a96c3d038fff9c8691af81e8a01de885cf6', 'research/benchmark/na-projects.json': '97e0a0a272046b858fa1e46b82f8b2b0afb3d81da3f0571d84edc3d0c58d7787', 'research/benchmark/projects-upstream.json': '54bfd0a005ae780525a6c8f5051f95a69671a40e989f0b2ddfba42c5ffc4d8c5', 'run.py': '340c51d7a90296e8f842ca7a2da7d8961499e5e620f5ab963667acf5e42dc928', 'run_research.py': 'aeea0208490def7517fb3f51d5354fdadd6cd4d9a97dcf5bd7c9ff643e65ae9e', 'scripts/check_upstream_fixture.py': '09503ddb82daf9533114ba28c1a8bfa8ad904b38c9cb2de9b97049e70db3881c', 'scripts/check_upstream_initialization.py': 'd72516f78e0c43814d5c742d0a6b7a414b7fbf94c0a52498209341484dbe1e29', 'setup_context.py': '44e697a8e6963ba58e19ad080ad4dd80289420c28d4f73eaf45e32548c3f9613', 'smoke_pilot.py': '39ae9ee5e07bddbc5ddce481335a1c1a56f67d9b210215e65f9bdf7c266d3853', 'tests/test_benchmark.py': 'fe51c6b1702d16e2dd2b24d13408fc752a5cd3f65ce3ac84b7211b5ffa9397a0', 'tests/test_council.py': 'a5a2a881fc6a02c85dfaada0919db6aadbe2c7d800738ad4c7a96629d9b23696', 'tests/test_multitarget.py': '3df432a208590398d39f3eb3b7f2305d35c13b375169ed65448ce79b2e22341c', 'tests/test_paired.py': '2387dfc5e20ca986c73857e86095e757f9d3609da954825c3d3de21c3973d169', 'tests/test_pilot.py': '2705fcd9bef011a5b3e0490483b43d600e1fc1b590bdac38c000de516bc59186', 'tests/test_prescreen.py': '0af81cea71e202fba5127a87243f48236a479641277f45f47588b3bb7d151833', 'tests/test_quality.py': 'b42b9de6dd07835b2e92ad6c04d24fca7aa89e48f93cb6311c97ff1d9ac2a665', 'tests/test_research.py': '9cf673d2d2511c655efbdcffa14a7b39a44a47a08c6b5fd97c61a8fea82e522c', 'worker.py': 'bb9a14655f9dbecddde4fa352e05895d31e61a8f9e1306b4c145665000676aae'}
PAYLOAD='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    assert set(archive.namelist())==set(HASHES)
    for name,expected in HASHES.items():
        data=archive.read(name)
        assert hashlib.sha256(data).hexdigest()==expected,name
        path=(WORKDIR/name).resolve()
        assert path.is_relative_to(WORKDIR.resolve())
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_bytes(data)
os.chdir(WORKDIR)
sys.path.insert(0,str(WORKDIR))
PROVENANCE=dict(base_commit=BASE_COMMIT,overlay_sha256=HASHES,source_kind='Git base plus embedded local changes; not claimed pushed')
print('Source:',WORKDIR,'Study:',ROOT)


In [ ]:
# Hardware detection precedes downloads. GPU count alone does not prove context fit.
import torch
assert torch.cuda.is_available() and torch.cuda.device_count()==2, 'Enable T4 x2 in Kaggle Settings'
print([(torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory/1024**3,2)) for i in range(2)])
subprocess.run([sys.executable,'-m','pip','install','uv==0.6.17'],check=True)
subprocess.run([sys.executable,'bootstrap_modern.py'],check=True,cwd=WORKDIR)
PYTHON='/tmp/test4py-modern/bin/python'
subprocess.run([PYTHON,'-m','pip','install','-r','requirements-paired.txt'],check=True,cwd=WORKDIR)
LOG=Path('/kaggle/working')/('UPSTREAM_LAUNCHER_'+STAMP+'.log')
def run_step(arguments, env=None):
    with LOG.open('a') as log:
        proc=subprocess.Popen([PYTHON,*arguments],cwd=WORKDIR,env=env,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        code=proc.wait()
    if ROOT.exists():
        shutil.copy2(LOG,ROOT/'notebook-launcher.log')
    if code:
        if ROOT.exists():
            from paired_study import make_bundle
            bundle=make_bundle(ROOT)
            from IPython.display import display,FileLink
            display(FileLink(str(bundle)))
        raise RuntimeError(f'Step exited {code}; retained artifacts at {ROOT}. Do not proceed to larger run.')


In [ ]:
# Offline code checks: hide GPUs to prevent accidental model inference in tests.
run_step(['-m','unittest','discover','-s','tests','-v'], env=dict(os.environ,CUDA_VISIBLE_DEVICES=''))
# Freeze budgets BEFORE inference. If stress later fails, use a new plan/output;
# never lower budgets for only one arm or silently retry another model.
run_step(['paired_study.py','--action','plan','--output',str(ROOT),
          '--max-input-tokens','4096','--max-new-tokens','512'])
(ROOT/'notebook-source.json').write_text(json.dumps(PROVENANCE,indent=2))
run_step(['paired_study.py','--action','prepare','--output',str(ROOT)])


In [ ]:
# Engineering fixtures: real upstream orchestration, SCRIPTED model/embedding.
# Their coverage/mutation numbers are not benchmark measurements.
preflight=json.loads((ROOT/'preflight.json').read_text())
for name,item in preflight.items():
    run_step(['scripts/check_upstream_initialization.py','--upstream',str(ROOT/'upstream'),
              '--source',item['import_root'],'--source-path',item['source_path'],
              '--output',str(ROOT/'engineering_fixture'/('init-'+name))],
             env=dict(os.environ,CUDA_VISIBLE_DEVICES=''))
for arm in ['baseline','improved']:
    run_step(['scripts/check_upstream_fixture.py','--upstream',str(ROOT/'upstream'),
              '--output',str(ROOT/'engineering_fixture'/arm),'--arm',arm],
             env=dict(os.environ,CUDA_VISIBLE_DEVICES=''))


In [ ]:
# Synthetic prefill + FORCED full-length decode at 2K/4K + 512 output.
# Repeat the largest case, require 1.5 GiB conservative memory headroom per GPU.
# This tests more than model loading; real upstream prompts remain budget-checked.
run_step(['paired_study.py','--action','stress','--output',str(ROOT),'--jobs','2'])


In [ ]:
# Small: genty.genty_repeat, one seed, one round, both arms; full package analyzed.
run_step(['paired_study.py','--action','small','--output',str(ROOT),'--jobs','2'])
from IPython.display import display,Markdown,FileLink
summary=json.loads((ROOT/'small-summary.json').read_text())
display(Markdown((ROOT/'small-report.md').read_text()))
print('Execution ready:',summary['execution_ready'],'Semantic review required:',summary['semantic_review_required'])
from paired_study import make_bundle
shutil.copy2(LOG,ROOT/'notebook-launcher.log')
display(FileLink(str(make_bundle(ROOT))))
print('Review raw tests, generation errors, resource budgets and locked fault pools before a larger run.')


## Optional larger paired pilot — not started by Run All
The locked larger plan has 3 modules in 2 projects (genty repeat/args and woa
whale_optimization), 2 seeds, 3 rounds, 2 arms: **12 module trials**. It is a
purpose-selected pilot, not representative of the 183-module paper benchmark.
The small module is development data. No significance/generalization claim.
Baseline failures are valid outcomes; engineering readiness is not all-suite pass.
Set the flag in the following cell only after reviewing the small archive.


In [ ]:
RUN_LARGER=True
if RUN_LARGER:
    small=json.loads((ROOT/'small-summary.json').read_text())
    assert small['execution_ready'], 'Small worker/hardware check incomplete'
    run_step(['paired_study.py','--action','run','--output',str(ROOT),'--jobs','2'])
    display(Markdown((ROOT/'main-report.md').read_text()))
    display(FileLink(str(make_bundle(ROOT))))
else:
    print('Larger run not launched. Small-run archive:',str(ROOT)+'.zip')
